In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)
import warnings
import logging

# Suppress MLflow warnings
warnings.filterwarnings('ignore')
logging.getLogger('mlflow').setLevel(logging.ERROR)


In [2]:
df = pd.read_csv('A:\project_iranian_churn\dataset\Customer Churn.csv')
df

,Call Failure,Complains,Subscription Length,Charge Amount,Seconds of Use,Frequency of use,Frequency of SMS,Distinct Called Numbers,Age Group,Tariff Plan,Status,Age,Customer Value,Churn
0,8,0,38,0,4370,71,5,17,3,1,1,30,197.640,0
1,0,0,39,0,318,5,7,4,2,1,2,25,46.035,0
2,10,0,37,0,2453,60,359,24,3,1,1,30,1536.520,0
3,10,0,38,0,4198,66,1,35,1,1,1,15,240.020,0
4,3,0,38,0,2393,58,2,33,1,1,1,15,145.805,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3145,21,0,19,2,6697,147,92,44,2,2,1,25,721.980,0
3146,17,0,17,1,9237,177,80,42,5,1,1,55,261.210,0
3147,13,0,18,4,3157,51,38,21,3,1,1,30,280.320,0
3148,7,0,11,2,4695,46,222,12,3,1,1,30,1077.640,0


In [3]:
df.head()

,Call Failure,Complains,Subscription Length,Charge Amount,Seconds of Use,Frequency of use,Frequency of SMS,Distinct Called Numbers,Age Group,Tariff Plan,Status,Age,Customer Value,Churn
0,8,0,38,0,4370,71,5,17,3,1,1,30,197.640,0
1,0,0,39,0,318,5,7,4,2,1,2,25,46.035,0
2,10,0,37,0,2453,60,359,24,3,1,1,30,1536.520,0
3,10,0,38,0,4198,66,1,35,1,1,1,15,240.020,0
4,3,0,38,0,2393,58,2,33,1,1,1,15,145.805,0


In [4]:
df = df.drop_duplicates(keep='first')

In [5]:
df.shape

(2850, 14)

In [6]:
df.columns = [col.strip().replace(' ', '_') for col in df.columns]
print(df.columns.tolist())

['Call__Failure', 'Complains', 'Subscription__Length', 'Charge__Amount', 'Seconds_of_Use', 'Frequency_of_use', 'Frequency_of_SMS', 'Distinct_Called_Numbers', 'Age_Group', 'Tariff_Plan', 'Status', 'Age', 'Customer_Value', 'Churn']


In [ ]:
churn_pct = df['Churn'].value_counts(normalize=True) * 100
print(churn_pct)

Churn
0    84.350877
1    15.649123
Name: proportion, dtype: float64


In [7]:
mlflow.set_tracking_uri("file:///A:/project_iranian_churn/mlruns")
mlflow.set_experiment("Iranian_Churn_Prediction")

print(f"✅ Tracking URI: {mlflow.get_tracking_uri()}")

✅ Tracking URI: file:///A:/project_iranian_churn/mlruns


In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

def run_experiment(nama_run, fitur_dipakai, model, df, 
                   target='Churn', handling='none'):
    """
    handling: 'none', 'class_weight', 'smote'
    """
    
    X = df[fitur_dipakai]
    y = df[target]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=180915, stratify=y
    )
    
    # Handling imbalance via class_weight
    if handling == 'class_weight':
        if hasattr(model, 'class_weight'):
            model.set_params(class_weight='balanced')
    
    # Buat pipeline
    if handling == 'smote':
        if isinstance(model, LogisticRegression):
            pipeline = ImbPipeline([
                ('smote',  SMOTE(random_state=42)),
                ('scaler', StandardScaler()),
                ('model',  model)
            ])
        else:
            pipeline = ImbPipeline([
                ('smote', SMOTE(random_state=42)),
                ('model', model)
            ])
    else:
        if isinstance(model, LogisticRegression):
            pipeline = Pipeline([
                ('scaler', StandardScaler()),
                ('model',  model)
            ])
        else:
            pipeline = Pipeline([
                ('model', model)
            ])
    
    with mlflow.start_run(run_name=nama_run):
        
        mlflow.log_param("fitur",          str(fitur_dipakai))
        mlflow.log_param("jumlah_fitur",   len(fitur_dipakai))
        mlflow.log_param("model",          type(model).__name__)
        mlflow.log_param("handling",       handling)
        mlflow.log_param("scaler", "StandardScaler" 
                         if isinstance(model, LogisticRegression) else "None")
        mlflow.log_params(model.get_params())
        
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]
        
        acc = accuracy_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1  = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob)
        
        mlflow.log_metric("accuracy",  acc)
        mlflow.log_metric("recall",    rec)
        mlflow.log_metric("f1_score",  f1)
        mlflow.log_metric("auc_roc",   auc)
        mlflow.sklearn.log_model(pipeline, "model")
        
        print(f"\n{'='*45}")
        print(f"Run      : {nama_run}")
        print(f"Handling : {handling}")
        print(f"Accuracy : {acc:.4f}")
        print(f"Recall   : {rec:.4f}")
        print(f"F1 Score : {f1:.4f}")
        print(f"AUC-ROC  : {auc:.4f}")
        print(f"{'='*45}")

print("✅ Fungsi siap!")

✅ Fungsi siap!


In [10]:
semua_fitur = [
    'Call__Failure', 'Complains', 'Subscription__Length',
    'Charge__Amount', 'Seconds_of_Use', 'Frequency_of_SMS',
    'Distinct_Called_Numbers', 'Age_Group', 'Tariff_Plan',
    'Status', 'Customer_Value'
]

fitur_mi_tinggi = [
    'Complains', 'Status', 'Seconds_of_Use',
    'Customer_Value', 'Charge__Amount'
]

eksperimen = [
    {
        "nama"  : "RF_semua_fitur",
        "fitur" : semua_fitur,
        "model" : RandomForestClassifier(n_estimators=100, random_state=180915)
    },
    {
        "nama"  : "RF_fitur_MI_tinggi",
        "fitur" : fitur_mi_tinggi,
        "model" : RandomForestClassifier(n_estimators=100, random_state=180915)
    },
    {
        "nama"  : "LogReg_semua_fitur",
        "fitur" : semua_fitur,
        "model" : LogisticRegression(max_iter=1000, random_state=180915)
    },
    {
        "nama"  : "LogReg_fitur_MI_tinggi",
        "fitur" : fitur_mi_tinggi,
        "model" : LogisticRegression(max_iter=1000, random_state=180915)
    },
]

for exp in eksperimen:
    run_experiment(exp["nama"], exp["fitur"], exp["model"], df)

print("\n✅ Semua eksperimen selesai! Buka http://127.0.0.1:5000")


Run      : RF_no_handling
Handling : none
Accuracy : 0.9193
Recall   : 0.6292
F1 Score : 0.7089
AUC-ROC  : 0.9592

Run      : RF_class_weight
Handling : class_weight
Accuracy : 0.9211
Recall   : 0.6404
F1 Score : 0.7170
AUC-ROC  : 0.9601

Run      : RF_smote
Handling : smote
Accuracy : 0.9105
Recall   : 0.8090
F1 Score : 0.7385
AUC-ROC  : 0.9532

Run      : LR_no_handling
Handling : none
Accuracy : 0.8860
Recall   : 0.3933
F1 Score : 0.5185
AUC-ROC  : 0.9105

Run      : LR_class_weight
Handling : class_weight
Accuracy : 0.8404
Recall   : 0.8427
F1 Score : 0.6224
AUC-ROC  : 0.9107

Run      : LR_smote
Handling : smote
Accuracy : 0.8000
Recall   : 0.8876
F1 Score : 0.5809
AUC-ROC  : 0.9104

✅ Semua eksperimen selesai!


In [ ]:
# %%
fitur_final = ['Complains', 'Status', 'Seconds_of_Use',
               'Customer_Value', 'Charge__Amount',
               'Subscription__Length', 'Frequency_of_SMS']

eksperimen = [
    # Random Forest
    {
        "nama"     : "RF_no_handling",
        "fitur"    : fitur_final,
        "model"    : RandomForestClassifier(n_estimators=100, random_state=42),
        "handling" : "none"
    },
    {
        "nama"     : "RF_class_weight",
        "fitur"    : fitur_final,
        "model"    : RandomForestClassifier(n_estimators=100, random_state=42),
        "handling" : "class_weight"
    },
    {
        "nama"     : "RF_smote",
        "fitur"    : fitur_final,
        "model"    : RandomForestClassifier(n_estimators=100, random_state=42),
        "handling" : "smote"
    },
    # Logistic Regression
    {
        "nama"     : "LR_no_handling",
        "fitur"    : fitur_final,
        "model"    : LogisticRegression(max_iter=1000, random_state=42),
        "handling" : "none"
    },
    {
        "nama"     : "LR_class_weight",
        "fitur"    : fitur_final,
        "model"    : LogisticRegression(max_iter=1000, random_state=42),
        "handling" : "class_weight"
    },
    {
        "nama"     : "LR_smote",
        "fitur"    : fitur_final,
        "model"    : LogisticRegression(max_iter=1000, random_state=42),
        "handling" : "smote"
    },
]

for exp in eksperimen:
    run_experiment(
        nama_run     = exp["nama"],
        fitur_dipakai = exp["fitur"],
        model        = exp["model"],
        df           = df,
        handling     = exp["handling"]
    )

print("\n✅ Semua eksperimen selesai!")

In [11]:
runs = mlflow.search_runs(experiment_names=["Iranian_Churn_Prediction"])
print(f"Jumlah run tersimpan: {len(runs)}")
print(runs[['tags.mlflow.runName', 'metrics.f1_score', 
            'metrics.recall', 'metrics.auc_roc']]\
     .sort_values('metrics.f1_score', ascending=False))

Jumlah run tersimpan: 10
      tags.mlflow.runName  metrics.f1_score  metrics.recall  metrics.auc_roc
9          RF_semua_fitur          0.802469        0.730337         0.984045
3                RF_smote          0.738462        0.808989         0.953199
4         RF_class_weight          0.716981        0.640449         0.960114
5          RF_no_handling          0.708861        0.629213         0.959238
8      RF_fitur_MI_tinggi          0.643275        0.617978         0.927877
1         LR_class_weight          0.622407        0.842697         0.910743
0                LR_smote          0.580882        0.887640         0.910369
7      LogReg_semua_fitur          0.555556        0.449438         0.916746
6  LogReg_fitur_MI_tinggi          0.542636        0.393258         0.906095
2          LR_no_handling          0.518519        0.393258         0.910486


Tracking URI saat ini: file:./mlruns
